In [1]:
# ============================================================================
# Cell 1: Imports (Enhanced)
# ============================================================================

import sys
sys.path.append('../src')
from data_loader import UniversalECGLoader
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.signal import resample
import neurokit2 as nk
from tqdm.auto import tqdm

print("✅ Imports complete")

✅ Imports complete


In [2]:
# ============================================================================
# Cell 2: Load Both AFDB and CINC17 Datasets
# ============================================================================

# Load AFDB
afdb_path = '../data/raw/mit-bih-afdb/'
afdb_loader = UniversalECGLoader(afdb_path)
afdb_records = afdb_loader.get_valid_records()

print(f"📊 AFDB: Found {len(afdb_records)} valid records")
print(f"   First 5: {afdb_records[:5]}\n")

# Load CINC17
cinc_path = '../data/raw/physionet-challenge-2017/training/'
cinc_loader = UniversalECGLoader(cinc_path)
cinc_records = cinc_loader.get_valid_records()

print(f"📊 CINC17: Found {len(cinc_records)} valid records")
print(f"   First 5: {cinc_records[:5]}\n")

# Load CINC17 labels
cinc_labels_path = Path('../data/raw/physionet-challenge-2017/REFERENCE-v3.csv')
cinc_labels_df = pd.read_csv(cinc_labels_path, header=None, names=['record_id', 'label'])
cinc_labels_dict = dict(zip(cinc_labels_df['record_id'], cinc_labels_df['label']))

print(f"📊 CINC17 Labels: {len(cinc_labels_dict)} records")
print(f"   Label distribution:")
label_dist = cinc_labels_df['label'].value_counts()
for label, count in label_dist.items():
    print(f"   - '{label}': {count} records")


ANALYZING DATASET AT: ..\data\raw\mit-bih-afdb

📁 Directory Structure:
   Root: ..\data\raw\mit-bih-afdb
   Subdirectories: old

📄 File Types Found:
   .atr: 50 files
   .atr-: 2 files
   .dat: 23 files
   .hea: 25 files
   .hea-: 23 files
   .qrs: 25 files
   .qrs-: 1 files
   .qrsc: 2 files
   .txt: 2 files
   .xws: 1 files

📊 Metadata Files:
   - notes.txt
   - SHA256SUMS.txt

🔍 Detected Format: WFDB
   Total record files: 23


✅ Found 25 records in WFDB format

🔍 Validating 25 records...
   Progress: 10/25 (8 valid)
   Progress: 20/25 (18 valid)

✅ Found 23 valid records
⚠️  Skipped 2 problematic records:
   • 00735: Failed to load record '00735': Error reading WFDB file: samp...
   • 03665: Failed to load record '03665': Error reading WFDB file: samp...
📊 AFDB: Found 23 valid records
   First 5: ['04015', '04043', '04048', '04126', '04746']


ANALYZING DATASET AT: ..\data\raw\physionet-challenge-2017\training

📁 Directory Structure:
   Root: ..\data\raw\physionet-challenge-2017\t

In [3]:
# ============================================================================
# Cell 3: Configuration and Common Preprocessing Functions
# ============================================================================

# Configuration
TARGET_FS = 250  # Target sampling frequency
WINDOW_SECONDS = 10  # Window duration
SAMPLES_PER_WINDOW = WINDOW_SECONDS * TARGET_FS  # 2500 samples

print(f"⚙️  Configuration:")
print(f"   Target sampling rate: {TARGET_FS} Hz")
print(f"   Window duration: {WINDOW_SECONDS} seconds")
print(f"   Samples per window: {SAMPLES_PER_WINDOW}")

# Common preprocessing function
def preprocess_and_clean_signal(signal, original_fs):
    """
    Resample and clean ECG signal.
    
    Args:
        signal: Raw ECG signal
        original_fs: Original sampling frequency
    
    Returns:
        Cleaned and resampled signal at TARGET_FS
    """
    # Step 1: Resample to target frequency
    if original_fs != TARGET_FS:
        num_samples = int(len(signal) * (TARGET_FS / original_fs))
        signal = resample(signal, num_samples)
    
    # Step 2: Clean the signal
    cleaned_signal = nk.ecg_clean(signal, sampling_rate=TARGET_FS)
    
    return cleaned_signal

print("\n✅ Preprocessing functions defined")

⚙️  Configuration:
   Target sampling rate: 250 Hz
   Window duration: 10 seconds
   Samples per window: 2500

✅ Preprocessing functions defined


In [4]:
# ============================================================================
# Cell 4: AFDB-Specific Functions (Pure Window Labeling)
# ============================================================================

def get_rhythm_at_sample_afdb(sample_idx, samples, aux_notes):
    """Get rhythm at a specific sample index for AFDB"""
    current_rhythm = 'N'
    
    for i, sample in enumerate(samples):
        if sample <= sample_idx:
            rhythm_str = str(aux_notes[i]) if i < len(aux_notes) else ''
            
            if '(AFIB' in rhythm_str or '(AFL' in rhythm_str:
                current_rhythm = 'A'
            elif '(N' in rhythm_str or rhythm_str == '':
                current_rhythm = 'N'
        else:
            break
    
    return current_rhythm


def get_pure_window_label_afdb(start_sample, end_sample, samples, aux_notes):
    """
    Label AFDB window ONLY if it contains a single rhythm throughout.
    Returns None if the window contains a rhythm transition.
    """
    # Step 1: Get rhythm at window start
    rhythm_at_start = get_rhythm_at_sample_afdb(start_sample, samples, aux_notes)
    
    # Step 2: Check if ANY rhythm changes occur INSIDE the window
    for change_sample in samples:
        if start_sample < change_sample < end_sample:
            # Transition detected inside window!
            return None  # This is a MIXED window
    
    # Step 3: Window is pure - return the rhythm
    return rhythm_at_start


def process_afdb_record(record_id, afdb_loader):
    """
    Process one AFDB record with pure window labeling.
    
    Returns:
        List of (window, label, metadata) tuples
    """
    windows_data = []
    
    try:
        # Load record
        record_data = afdb_loader.load_record(record_id)
        
        if 'annotations' not in record_data:
            return windows_data
        
        annotations = record_data['annotations']
        
        # Find annotation keys
        sample_key = None
        aux_key = None
        for key in annotations.keys():
            if 'sample' in key.lower():
                sample_key = key
            if 'aux' in key.lower() or 'note' in key.lower():
                aux_key = key
        
        if not sample_key or not aux_key:
            return windows_data
        
        samples = annotations[sample_key]
        aux_notes = annotations[aux_key]
        
        # Get signal (first lead)
        signal = record_data['signal'][:, 0] if record_data['signal'].ndim > 1 else record_data['signal']
        
        # Preprocess signal
        signal = preprocess_and_clean_signal(signal, record_data['fs'])
        
        # Extract windows
        num_windows = len(signal) // SAMPLES_PER_WINDOW
        scale_factor = TARGET_FS / record_data['fs']
        
        for window_idx in range(num_windows):
            start_sample = int((window_idx * SAMPLES_PER_WINDOW) / scale_factor)
            end_sample = int(((window_idx + 1) * SAMPLES_PER_WINDOW) / scale_factor)
            
            # Get pure window label
            label = get_pure_window_label_afdb(start_sample, end_sample, samples, aux_notes)
            
            if label is not None and label in ['A', 'N']:
                window_start_idx = window_idx * SAMPLES_PER_WINDOW
                window_end_idx = window_start_idx + SAMPLES_PER_WINDOW
                window = signal[window_start_idx:window_end_idx]
                
                if len(window) == SAMPLES_PER_WINDOW:
                    windows_data.append({
                        'window': window,
                        'label': label,
                        'source': 'AFDB',
                        'record_id': record_id,
                        'window_idx': window_idx,
                        'is_pure': True
                    })
    
    except Exception as e:
        print(f"❌ Error processing AFDB record {record_id}: {str(e)[:60]}")
    
    return windows_data

print("✅ AFDB processing functions defined")

✅ AFDB processing functions defined


In [5]:
# ============================================================================
# Cell 5: CINC17-Specific Functions (Pragmatic Approach)
# ============================================================================

def process_cinc17_record(record_id, cinc_loader, label):
    """
    Process one CINC17 record - chop into 10s windows.
    All windows get the SAME label (pragmatic approach).
    
    Args:
        record_id: Record identifier
        cinc_loader: CINC17 data loader
        label: Record label ('A', 'N', 'O', or '~')
    
    Returns:
        List of (window, label, metadata) tuples
    """
    windows_data = []
    
    # Only process AFib and Normal
    if label not in ['A', 'N']:
        return windows_data
    
    try:
        # Load record
        record_data = cinc_loader.load_record(record_id)
        
        # Get signal (first lead if multiple)
        signal = record_data['signal'][:, 0] if record_data['signal'].ndim > 1 else record_data['signal']
        
        # Preprocess signal
        signal = preprocess_and_clean_signal(signal, record_data['fs'])
        
        # Extract non-overlapping 10-second windows
        num_windows = len(signal) // SAMPLES_PER_WINDOW
        
        for window_idx in range(num_windows):
            start_idx = window_idx * SAMPLES_PER_WINDOW
            end_idx = start_idx + SAMPLES_PER_WINDOW
            
            window = signal[start_idx:end_idx]
            
            if len(window) == SAMPLES_PER_WINDOW:
                # Track if window is at edge (for later analysis)
                is_edge = (window_idx == 0 or window_idx == num_windows - 1)
                
                windows_data.append({
                    'window': window,
                    'label': label,
                    'source': 'CINC17',
                    'record_id': record_id,
                    'window_idx': window_idx,
                    'total_windows': num_windows,
                    'is_edge': is_edge,
                    'is_pure': None  # Unknown for CINC17
                })
    
    except Exception as e:
        print(f"❌ Error processing CINC17 record {record_id}: {str(e)[:60]}")
    
    return windows_data

print("✅ CINC17 processing functions defined")

✅ CINC17 processing functions defined


In [6]:
# ============================================================================
# Cell 6: Process All AFDB Records
# ============================================================================

print(f"\n{'='*70}")
print(f"PROCESSING ALL AFDB RECORDS")
print(f"{'='*70}\n")

afdb_windows_data = []
afdb_stats = {
    'total_records': 0,
    'successful_records': 0,
    'total_windows': 0,
    'kept_windows': 0,
    'discarded_windows': 0,
    'afib_windows': 0,
    'normal_windows': 0
}

for record_id in tqdm(afdb_records, desc="Processing AFDB"):
    windows = process_afdb_record(record_id, afdb_loader)
    
    if windows:
        afdb_stats['successful_records'] += 1
        afdb_windows_data.extend(windows)

afdb_stats['total_records'] = len(afdb_records)
afdb_stats['kept_windows'] = len(afdb_windows_data)

# Count labels
for window_data in afdb_windows_data:
    if window_data['label'] == 'A':
        afdb_stats['afib_windows'] += 1
    elif window_data['label'] == 'N':
        afdb_stats['normal_windows'] += 1

print(f"\n📊 AFDB Processing Summary:")
print(f"   Records processed: {afdb_stats['successful_records']}/{afdb_stats['total_records']}")
print(f"   Total kept windows: {afdb_stats['kept_windows']:,}")
print(f"   - AFib windows: {afdb_stats['afib_windows']:,} ({afdb_stats['afib_windows']/afdb_stats['kept_windows']*100:.1f}%)")
print(f"   - Normal windows: {afdb_stats['normal_windows']:,} ({afdb_stats['normal_windows']/afdb_stats['kept_windows']*100:.1f}%)")


PROCESSING ALL AFDB RECORDS



Processing AFDB: 100%|██████████| 23/23 [00:18<00:00,  1.26it/s]


📊 AFDB Processing Summary:
   Records processed: 23/23
   Total kept windows: 83,750
   - AFib windows: 33,945 (40.5%)
   - Normal windows: 49,805 (59.5%)


In [7]:
# ============================================================================
# Cell 7: Process All CINC17 Records
# ============================================================================

print(f"\n{'='*70}")
print(f"PROCESSING ALL CINC17 RECORDS")
print(f"{'='*70}\n")

cinc17_windows_data = []
cinc17_stats = {
    'total_records': 0,
    'processed_records': 0,
    'skipped_records': 0,
    'total_windows': 0,
    'afib_windows': 0,
    'normal_windows': 0,
    'edge_windows': 0,
    'middle_windows': 0
}

# Count records by label type
label_type_counts = {'A': 0, 'N': 0, 'O': 0, '~': 0, 'other': 0}

for record_id in tqdm(cinc_records, desc="Processing CINC17"):
    # Get clean record ID
    clean_id = Path(record_id).stem
    label = cinc_labels_dict.get(clean_id)
    
    if label:
        label_type_counts[label if label in ['A', 'N', 'O', '~'] else 'other'] += 1
        
        if label in ['A', 'N']:
            windows = process_cinc17_record(record_id, cinc_loader, label)
            
            if windows:
                cinc17_stats['processed_records'] += 1
                cinc17_windows_data.extend(windows)
                
                # Count edge vs middle windows
                for window_data in windows:
                    if window_data['is_edge']:
                        cinc17_stats['edge_windows'] += 1
                    else:
                        cinc17_stats['middle_windows'] += 1
        else:
            cinc17_stats['skipped_records'] += 1

cinc17_stats['total_records'] = len(cinc_records)
cinc17_stats['total_windows'] = len(cinc17_windows_data)

# Count labels
for window_data in cinc17_windows_data:
    if window_data['label'] == 'A':
        cinc17_stats['afib_windows'] += 1
    elif window_data['label'] == 'N':
        cinc17_stats['normal_windows'] += 1

print(f"\n📊 CINC17 Processing Summary:")
print(f"   Total records: {cinc17_stats['total_records']:,}")
print(f"   Processed (A/N): {cinc17_stats['processed_records']:,}")
print(f"   Skipped (O/~): {cinc17_stats['skipped_records']:,}")
print(f"\n   Label distribution in dataset:")
for label_type, count in sorted(label_type_counts.items()):
    print(f"   - '{label_type}': {count:,} records")

print(f"\n   Total windows created: {cinc17_stats['total_windows']:,}")
print(f"   - AFib windows: {cinc17_stats['afib_windows']:,} ({cinc17_stats['afib_windows']/cinc17_stats['total_windows']*100:.1f}%)")
print(f"   - Normal windows: {cinc17_stats['normal_windows']:,} ({cinc17_stats['normal_windows']/cinc17_stats['total_windows']*100:.1f}%)")
print(f"\n   Window positions:")
print(f"   - Edge windows: {cinc17_stats['edge_windows']:,} ({cinc17_stats['edge_windows']/cinc17_stats['total_windows']*100:.1f}%)")
print(f"   - Middle windows: {cinc17_stats['middle_windows']:,} ({cinc17_stats['middle_windows']/cinc17_stats['total_windows']*100:.1f}%)")


PROCESSING ALL CINC17 RECORDS



Processing CINC17: 100%|██████████| 8828/8828 [00:15<00:00, 571.93it/s]


📊 CINC17 Processing Summary:
   Total records: 8,828
   Processed (A/N): 6,018
   Skipped (O/~): 2,799

   Label distribution in dataset:
   - 'A': 805 records
   - 'N': 5,224 records
   - 'O': 2,480 records
   - 'other': 0 records
   - '~': 319 records

   Total windows created: 18,859
   - AFib windows: 2,534 (13.4%)
   - Normal windows: 16,325 (86.6%)

   Window positions:
   - Edge windows: 11,642 (61.7%)
   - Middle windows: 7,217 (38.3%)


In [8]:
# ============================================================================
# Cell 8: Combine Both Datasets and Analyze
# ============================================================================

print(f"\n{'='*70}")
print(f"COMBINING DATASETS")
print(f"{'='*70}\n")

# Combine all windows
all_windows_data = afdb_windows_data + cinc17_windows_data

print(f"📊 Combined Dataset Summary:")
print(f"   Total windows: {len(all_windows_data):,}")
print(f"   - From AFDB: {len(afdb_windows_data):,} ({len(afdb_windows_data)/len(all_windows_data)*100:.1f}%)")
print(f"   - From CINC17: {len(cinc17_windows_data):,} ({len(cinc17_windows_data)/len(all_windows_data)*100:.1f}%)")

# Extract arrays
X = np.array([w['window'] for w in all_windows_data])
y = np.array([w['label'] for w in all_windows_data])

print(f"\n   Data shape: {X.shape}")
print(f"   Labels shape: {y.shape}")

# Count by label
afib_count = np.sum(y == 'A')
normal_count = np.sum(y == 'N')

print(f"\n   Overall Label Distribution:")
print(f"   - AFib: {afib_count:,} ({afib_count/len(y)*100:.1f}%)")
print(f"   - Normal: {normal_count:,} ({normal_count/len(y)*100:.1f}%)")
print(f"   - Class ratio (Normal:AFib): {normal_count/afib_count:.2f}:1")

# Analyze by source
print(f"\n📊 Breakdown by Source:")
print(f"\n   AFDB:")
afdb_afib = sum(1 for w in afdb_windows_data if w['label'] == 'A')
afdb_normal = sum(1 for w in afdb_windows_data if w['label'] == 'N')
print(f"   - AFib: {afdb_afib:,} ({afdb_afib/len(afdb_windows_data)*100:.1f}%)")
print(f"   - Normal: {afdb_normal:,} ({afdb_normal/len(afdb_windows_data)*100:.1f}%)")

print(f"\n   CINC17:")
cinc_afib = sum(1 for w in cinc17_windows_data if w['label'] == 'A')
cinc_normal = sum(1 for w in cinc17_windows_data if w['label'] == 'N')
print(f"   - AFib: {cinc_afib:,} ({cinc_afib/len(cinc17_windows_data)*100:.1f}%)")
print(f"   - Normal: {cinc_normal:,} ({cinc_normal/len(cinc17_windows_data)*100:.1f}%)")


COMBINING DATASETS

📊 Combined Dataset Summary:
   Total windows: 102,609
   - From AFDB: 83,750 (81.6%)
   - From CINC17: 18,859 (18.4%)

   Data shape: (102609, 2500)
   Labels shape: (102609,)

   Overall Label Distribution:
   - AFib: 36,479 (35.6%)
   - Normal: 66,130 (64.4%)
   - Class ratio (Normal:AFib): 1.81:1

📊 Breakdown by Source:

   AFDB:
   - AFib: 33,945 (40.5%)
   - Normal: 49,805 (59.5%)

   CINC17:
   - AFib: 2,534 (13.4%)
   - Normal: 16,325 (86.6%)


In [9]:
# ============================================================================
# Cell 9: Create Detailed Analysis DataFrame
# ============================================================================

# Create DataFrame for detailed analysis
df_analysis = pd.DataFrame(all_windows_data)
df_analysis = df_analysis.drop('window', axis=1)  # Remove large array for display

print(f"\n{'='*70}")
print(f"DETAILED ANALYSIS")
print(f"{'='*70}\n")

print("First 10 rows:")
print(df_analysis.head(10))

print(f"\n📊 Statistics by Source:")
print(df_analysis.groupby('source')['label'].value_counts())

print(f"\n📊 CINC17 Edge vs Middle Analysis:")
if 'is_edge' in df_analysis.columns:
    cinc_df = df_analysis[df_analysis['source'] == 'CINC17']
    if len(cinc_df) > 0:
        print(cinc_df.groupby(['is_edge', 'label']).size())


DETAILED ANALYSIS

First 10 rows:
  label source record_id  window_idx is_pure  total_windows is_edge
0     N   AFDB     04015           1    True            NaN     NaN
1     N   AFDB     04015           2    True            NaN     NaN
2     N   AFDB     04015           3    True            NaN     NaN
3     N   AFDB     04015           4    True            NaN     NaN
4     N   AFDB     04015           5    True            NaN     NaN
5     N   AFDB     04015           6    True            NaN     NaN
6     N   AFDB     04015           7    True            NaN     NaN
7     N   AFDB     04015           8    True            NaN     NaN
8     N   AFDB     04015           9    True            NaN     NaN
9     N   AFDB     04015          10    True            NaN     NaN

📊 Statistics by Source:
source  label
AFDB    N        49805
        A        33945
CINC17  N        16325
        A         2534
Name: count, dtype: int64

📊 CINC17 Edge vs Middle Analysis:
is_edge  label
False    A

In [10]:
# ============================================================================
# Cell 10: Save Processed Dataset
# ============================================================================

print(f"\n{'='*70}")
print(f"SAVING PROCESSED DATASET")
print(f"{'='*70}\n")

# Create output directory
processed_dir = Path('../data/processed/')
processed_dir.mkdir(exist_ok=True, parents=True)

# Save arrays
np.save(processed_dir / 'X_combined.npy', X)
np.save(processed_dir / 'y_combined.npy', y)

# Save metadata
metadata_df = pd.DataFrame([
    {k: v for k, v in w.items() if k != 'window'} 
    for w in all_windows_data
])
metadata_df.to_csv(processed_dir / 'metadata.csv', index=False)

# Save summary statistics
summary_stats = {
    'total_windows': len(all_windows_data),
    'afdb_windows': len(afdb_windows_data),
    'cinc17_windows': len(cinc17_windows_data),
    'afib_windows': int(afib_count),
    'normal_windows': int(normal_count),
    'window_shape': X.shape,
    'sampling_rate': TARGET_FS,
    'window_duration': WINDOW_SECONDS
}

import json
with open(processed_dir / 'dataset_summary.json', 'w') as f:
    json.dump(summary_stats, f, indent=2)

print(f"✅ Saved processed dataset to {processed_dir}")
print(f"\n   Files created:")
print(f"   - X_combined.npy: {X.shape} ({X.nbytes / 1e6:.1f} MB)")
print(f"   - y_combined.npy: {y.shape} ({y.nbytes / 1e6:.1f} MB)")
print(f"   - metadata.csv: {len(metadata_df)} rows")
print(f"   - dataset_summary.json")

print(f"\n🎉 Processing complete!")
print(f"\n   Final dataset ready for model training:")
print(f"   - Total samples: {len(X):,}")
print(f"   - Input shape: {X.shape}")
print(f"   - AFib: {afib_count:,} ({afib_count/len(y)*100:.1f}%)")
print(f"   - Normal: {normal_count:,} ({normal_count/len(y)*100:.1f}%)")


SAVING PROCESSED DATASET

✅ Saved processed dataset to ..\data\processed

   Files created:
   - X_combined.npy: (102609, 2500) (2052.2 MB)
   - y_combined.npy: (102609,) (0.4 MB)
   - metadata.csv: 102609 rows
   - dataset_summary.json

🎉 Processing complete!

   Final dataset ready for model training:
   - Total samples: 102,609
   - Input shape: (102609, 2500)
   - AFib: 36,479 (35.6%)
   - Normal: 66,130 (64.4%)
